# Data Science Project 1: Data Wrangling Practical
**Duration:** 1 Hour, 15 Minutes  
**Format:** Open-notes code implementation (Python/Pandas)

### **Project Scenario**
You have been given a raw slice of **Glassdoor Data Science Job Postings**. The dataset contains a variety of small errors, missing variables, and split locations. Your goal is to clean this main dataset, create engineered features, and merge it with a regional mapping file so the analytics team can use it.

## Load the data
Download the data from the Blackboard to your computer frist, then load your libraries and the datasets required for this exam here.

In [5]:
# import all libaries and load your datasets your here

import numpy as np
import pandas as pd
import math

glassDoor = pd.read_csv('glassdoor_jobs.csv')
stateRegions = pd.read_csv('state_regions.csv')


## Part 1: Missing Data & Filtering (30 Points)

### **Task 1.1 (Standardizing Nulls)**
The `Rating` column represents missing values as `-1`. Replace all occurrences of `-1` with the dataset's overall **average (mean)** rating value.

In [17]:
# Write your solution for Task 1.1 here

glassDoor.replace({'Rating':-1.0}, np.nan, inplace=True)

ratingMean = glassDoor['Rating'].mean()
print(f"Rating mean: {ratingMean}")

glassDoor.fillna({'Rating':ratingMean}, inplace=True)


Rating mean: 3.8857664233576643


,Job_Title,Job Description,Rating,Company_Name,Location,Founded,Industry
0,Sr Data Scientist,Description\n\nThe Senior Data Scientist is re...,3.100000,Healthfirst\n3.1,"New York, NY",1993,Insurance Carriers
1,Data Scientist,"Secure our Nation, Ignite your Future\n\nJoin ...",4.200000,ManTech\n4.2,"Chantilly, VA",1968,Research & Development
2,Data Scientist,Overview\n\n\nAnalysis Group is one of the lar...,3.885766,Analysis Group\n3.8,"Boston, MA",1981,Consulting
3,Data Scientist,JOB DESCRIPTION:\n\nDo you have a passion for ...,3.500000,INFICON\n3.5,"Newton, MA",2000,Electrical & Electronic Manufacturing
4,Data Scientist,Data Scientist\nAffinity Solutions / Marketing...,2.900000,Affinity Solutions\n2.9,"New York, NY",1998,Advertising & Marketing
...,...,...,...,...,...,...,...
667,Data Scientist,Summary\n\nWe’re looking for a data scientist ...,3.885766,TRANZACT\n3.6,"Fort Lee, NJ",1989,Advertising & Marketing
668,Data Scientist,Job Description\nBecome a thought leader withi...,3.885766,JKGT,"San Francisco, CA",-1,-1
669,Data Scientist,Join a thriving company that is changing the w...,3.885766,AccessHope,"Irwindale, CA",-1,-1
670,Data Scientist,100 Remote Opportunity As an AINLP Data Scient...,5.000000,ChaTeck Incorporated\n5.0,"San Francisco, CA",-1,Advertising & Marketing


### **Task 1.2 (Handling Blanks)**
The `Industry` column has several completely blank or empty cells. Fill these missing values with the string text `"Unknown"`.

In [21]:
# Write your solution for Task 1.2 here

glassDoor.fillna({'Industry':"Unknown"}, inplace=True)
glassDoor.head(10)


,Job_Title,Job Description,Rating,Company_Name,Location,Founded,Industry
0,Sr Data Scientist,Description\n\nThe Senior Data Scientist is re...,3.100000,Healthfirst\n3.1,"New York, NY",1993,Insurance Carriers
1,Data Scientist,"Secure our Nation, Ignite your Future\n\nJoin ...",4.200000,ManTech\n4.2,"Chantilly, VA",1968,Research & Development
2,Data Scientist,Overview\n\n\nAnalysis Group is one of the lar...,3.885766,Analysis Group\n3.8,"Boston, MA",1981,Consulting
3,Data Scientist,JOB DESCRIPTION:\n\nDo you have a passion for ...,3.500000,INFICON\n3.5,"Newton, MA",2000,Electrical & Electronic Manufacturing
4,Data Scientist,Data Scientist\nAffinity Solutions / Marketing...,2.900000,Affinity Solutions\n2.9,"New York, NY",1998,Advertising & Marketing
5,Data Scientist,About Us:\n\nHeadquartered in beautiful Santa ...,4.200000,HG Insights\n4.2,"Santa Barbara, CA",2010,Unknown
6,Data Scientist / Machine Learning Expert,Posting Title\nData Scientist / Machine Learni...,3.900000,Novartis\n3.9,"Cambridge, MA",1996,Biotech & Pharmaceuticals
7,Data Scientist,Introduction\n\nHave you always wanted to run ...,3.500000,iRobot\n3.5,"Bedford, MA",1990,Consumer Electronics & Appliances Stores
8,Staff Data Scientist - Analytics,Intuit is seeking a Staff Data Scientist to co...,4.400000,Intuit - Data\n4.4,"San Diego, CA",1983,Computer Hardware & Software
9,Data Scientist,Ready to write the best chapter of your career...,3.600000,XSELL Technologies\n3.6,"Chicago, IL",2014,Unknown


### **Task 1.3 (Data Filtering)**
A few rows contain placeholder or invalid values in the `Founded` year column (values less than or equal to `0`). **Filter the dataset** to permanently remove these invalid rows.

In [33]:
# Write your solution for Task 1.3 here

print(glassDoor.loc[68:70,'Founded'])
glassDoor.dropna(axis=0, subset='Founded', inplace=True)

print(glassDoor.loc[68:70,'Founded'])


68    2015
69      -1
70    2006
Name: Founded, dtype: int64
68    2015
69      -1
70    2006
Name: Founded, dtype: int64


## Part 2: String Manipulations (20 Points)

### **Task 2.1 (Location Splitting)**
The `Location` column lists locations as `"City, State"` (e.g., `"Chicago, IL"` or `"Austin, TX"`). Split this column into two separate, brand-new columns: `Job_City` and `Job_State`.

In [ ]:
# Write your solution for Task 2.1 here

glassDoor['Job_City'] = glassDoor['Location'].str.extract(r"")
glassDoor['Job_State'] = glassDoor['Location'].str.extract(r"$")


### **Task 2.2 (Whitespace Cleaning)**
The new `Job_State` column might contain accidental leading or trailing empty spaces from the split. Apply a string cleaning function (like `.str.strip()`) to ensure no accidental spaces remain.

In [36]:
# Write your solution for Task 2.2 here

glassDoor['Job_State'].str.strip()


## Part 3: Feature Engineering with Grouped Transformations (20 Points)

### **Task 3.1 (Grouped Transform)**
The recruiting team wants to identify jobs that offer a higher-than-average user rating relative to their specific field. 

Use `.groupby()` and `.transform()` to calculate the **average (mean) rating for each `Industry`**, broadcasting it back as a new column named `Industry_Avg_Rating`. Then, create a column called `Rating_Above_Industry_Avg` by subtracting `Industry_Avg_Rating` from the job's original `Rating`.

In [38]:
# Write your solution for Task 3.1 here

glassDoor['Industry_Avg_Rating'] = glassDoor.groupby('Industry')['Rating'].transform('mean')
glassDoor['Rating_Above_Industry_Avg'] = glassDoor['Rating'] - glassDoor['Industry_Avg_Rating']

glassDoor.head(15)


,Job_Title,Job Description,Rating,Company_Name,Location,Founded,Industry,Industry_Avg_Rating,Rating_Above_Industry_Avg
0,Sr Data Scientist,Description\n\nThe Senior Data Scientist is re...,3.100000,Healthfirst\n3.1,"New York, NY",1993,Insurance Carriers,3.792322,-0.692322
1,Data Scientist,"Secure our Nation, Ignite your Future\n\nJoin ...",4.200000,ManTech\n4.2,"Chantilly, VA",1968,Research & Development,3.904471,0.295529
2,Data Scientist,Overview\n\n\nAnalysis Group is one of the lar...,3.885766,Analysis Group\n3.8,"Boston, MA",1981,Consulting,3.874531,0.011235
3,Data Scientist,JOB DESCRIPTION:\n\nDo you have a passion for ...,3.500000,INFICON\n3.5,"Newton, MA",2000,Electrical & Electronic Manufacturing,3.692883,-0.192883
4,Data Scientist,Data Scientist\nAffinity Solutions / Marketing...,2.900000,Affinity Solutions\n2.9,"New York, NY",1998,Advertising & Marketing,3.731300,-0.831300
5,Data Scientist,About Us:\n\nHeadquartered in beautiful Santa ...,4.200000,HG Insights\n4.2,"Santa Barbara, CA",2010,Unknown,3.826730,0.373270
6,Data Scientist / Machine Learning Expert,Posting Title\nData Scientist / Machine Learni...,3.900000,Novartis\n3.9,"Cambridge, MA",1996,Biotech & Pharmaceuticals,3.694938,0.205062
7,Data Scientist,Introduction\n\nHave you always wanted to run ...,3.500000,iRobot\n3.5,"Bedford, MA",1990,Consumer Electronics & Appliances Stores,3.500000,0.000000
8,Staff Data Scientist - Analytics,Intuit is seeking a Staff Data Scientist to co...,4.400000,Intuit - Data\n4.4,"San Diego, CA",1983,Computer Hardware & Software,4.268737,0.131263
9,Data Scientist,Ready to write the best chapter of your career...,3.600000,XSELL Technologies\n3.6,"Chicago, IL",2014,Unknown,3.826730,-0.226730


## Part 4: Merging & Aggregating Data (30 Points)

### **Task 4.1 (Dataset Merging)**
You are provided a second, clean reference CSV file named `state_regions.csv` which contains two columns: `State_Abbreviation` and `US_Region` (e.g., *IL, Midwest*). 

Perform a **left merge** to add the `US_Region` column to your main job postings dataframe by matching your new `Job_State` column with the reference file's `State_Abbreviation` column.

In [ ]:
# Write your solution for Task 4.1 here
glassDoor.merge(stateRegions['US_Region'], on='', how='left')

### **Task 4.2 (Grouped Aggregation)**
Group your newly merged dataframe by the `US_Region` column. For each region, calculate:
1. The **total count** of job postings in that region.
2. The **average (mean)** employee `Rating` in that region.

In [ ]:
# Write your solution for Task 4.2 here


### **Task 4.3 (Data Export)**
Save your final, aggregated summary table from Task 4.2 to a new file named `regional_job_summary.csv`.

In [ ]:
# Write your solution for Task 4.3 here